# 02 · Data contracts: OOP, typing, Pydantic and decorators
**⏱ 12 minutes · From Java records and interfaces to AI tool schemas**

Imagine an AI service that classifies a support request and proposes a response. It's easy to pass around a dictionary. But what if the model returns the wrong shape? What if the ID is negative, or the priority is `"super-important"` when your application accepts only three values?

Your Java experience gives you the right instincts: **DTOs, validation, interfaces, and clear contracts**. Python has these too, with some important differences.

## 1. Java records, Python dataclasses, and runtime validation

**Java:**

```java
public record Ticket(int id, String subject) {}

public record ValidatedTicket(
    @Positive int id,
    @NotBlank String subject
) {}
```

A Java record reduces boilerplate. Bean Validation annotations require a validator to run; the annotations alone don't enforce constraints.

**Python:** `@dataclass` generates routine container methods. A type hint is guidance for tools such as IDEs, **not** a runtime validator. Pydantic `BaseModel` **validates data at model creation**, making it useful at API and LLM-output boundaries.

**Prediction:** will a normal dataclass reject `id=-1`?

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel, Field, ValidationError

@dataclass
class DraftTicket:
    id: int
    subject: str

class TicketInput(BaseModel):
    id: int = Field(gt=0)
    subject: str = Field(min_length=3)
    priority: str = "normal"

plain = DraftTicket(id=-1, subject="x")
print("Dataclass accepted:", plain)
validated = TicketInput(id=12, subject="Login problem")
print("Pydantic produced:", validated.model_dump())

## 2. Let validation fail on purpose

An LLM can produce syntactically valid JSON that breaks business constraints. We can catch that before acting on it.

The Pydantic object below violates **two constraints**. We catch the `ValidationError` deliberately and inspect the failing fields. In real applications, don't print personally identifiable input or secrets when reporting validation failures.

> **Try:** fix only one field and see how the error list changes.

In [ ]:
try:
    TicketInput(id=-5, subject="x")
except ValidationError as error:
    print("Rejected fields:", [problem["loc"] for problem in error.errors()])
else:
    raise AssertionError("Expected invalid data")

## 3. Java enum → Python Literal (challenge)

A Java `enum` constrains a set of permitted values. Python `Literal` tells typing tools which values are expected; combined with **Pydantic**, it also becomes runtime-validated.

**Your turn:** change `"high"` to `"urgent"`, run the cell, understand what happens, then restore the correct value.

<details><summary>Hint</summary>Look for the `Literal[...]` declaration; Pydantic rejects strings outside that list.</details>

In [ ]:
from typing import Literal

class StrictTicket(BaseModel):
    id: int = Field(gt=0)
    priority: Literal["low", "normal", "high"] = "normal"

sample = StrictTicket(id=42, priority="high")
assert sample.priority == "high"
print(sample.model_dump())

## 4. Java interfaces vs Python Protocol

**Java:**

```java
interface Formatter { String format(int ticketId); }
class DefaultFormatter implements Formatter {
    public String format(int id) { return "T-" + id; }
}
```

In Python, `Protocol` supports **structural typing**: a class matches the contract if it has the right methods. It needn't explicitly declare `implements`. This is useful when substituting tool implementations or test doubles.

A static type checker may flag mismatches. The annotation alone doesn't automatically enforce the interface at runtime.

In [ ]:
from typing import Protocol

class Formatter(Protocol):
    def format(self, ticket_id: int) -> str: ...

class DefaultFormatter:
    def format(self, ticket_id: int) -> str:
        return f"T-{ticket_id}"

formatter: Formatter = DefaultFormatter()
assert formatter.format(10) == "T-10"
print(formatter.format(10))

## 5. Decorators are executable, not just metadata

Java annotations can give frameworks information that they inspect later. In Python, a **decorator runs at function definition time**. It receives a function and may *wrap* or *register* that function.

When you see `@trace_call` below, read it as `lookup_ticket = trace_call(lookup_ticket)`.

**Prediction:** does the decorator change the return value? What prints first?

In [ ]:
from functools import wraps

def trace_call(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("Calling:", function.__name__)
        return function(*args, **kwargs)
    return wrapper

@trace_call
def lookup_ticket(ticket_id: int) -> str:
    return f"Ticket {ticket_id}"

assert lookup_ticket(11) == "Ticket 11"

## 6. Why agent frameworks use tool decorators

Many agent frameworks register read-only and write-capable tools with a decorator. Here's a tiny **read-only registry** built with Python alone. This is not a full agent and does not make any API call.

**Your turn:** add a second function using `@tool`. What keys should appear in the `tools` dictionary?

In [ ]:
from collections.abc import Callable

tools: dict[str, Callable[..., str]] = {}

def tool(function: Callable[..., str]):
    tools[function.__name__] = function
    return function

@tool
def get_ticket_status(ticket_id: int) -> str:
    return "open" if ticket_id > 0 else "unknown"

assert "get_ticket_status" in tools
assert tools["get_ticket_status"](10) == "open"
print("Registered tool names:", list(tools))

### The four contracts you should remember

1. **Dataclass:** stores structured objects conveniently.
2. **Type hints / Protocol:** describe code contracts for editors and type checkers.
3. **Pydantic:** checks untrusted structured input **at runtime**.
4. **Decorator:** transforms or registers a callable; tool registration is **not authorization** to execute a dangerous action.

**Checkpoint:** which one would you use to reject `priority="urgent"` coming back from an LLM? Why?

**Next:** [03 · Async and streaming](03_async_streaming.ipynb). See [Pydantic and decorators](../guides/04_pydantic_and_decorators.md) for a slower explanation.